# Session 1: Foundations of LLMs & Prompt Engineering
**Duration:** 1.5 Hours | **Day 1** | **Hands-On Workshop**

---

## Learning Objectives
By the end of this session, participants will:
1. Understand the evolution from NLP to Transformers to LLMs
2. Grasp the core architecture of Transformers and Attention mechanism
3. Differentiate between Generative AI and Traditional ML
4. Design effective prompts using zero-shot, few-shot, and system prompts
5. Control LLM outputs using temperature, max tokens, and formatting

---

## Prerequisites
- Ollama installed and running (`ollama serve`)
- Model pulled: `ollama pull llama3.2`
- Python packages: `pip install langchain langchain-ollama requests`

---
## Part 1: Evolution of NLP to LLMs (Theory - 20 min)

### 1.1 The Journey: NLP --> Transformers --> LLMs

**Era 1: Rule-Based NLP (1950s-1990s)**
- Hand-crafted grammar rules
- Pattern matching (regex)
- Limitation: Could not handle ambiguity or scale

**Era 2: Statistical NLP (1990s-2010s)**
- Bag of Words, TF-IDF
- Hidden Markov Models, Naive Bayes
- Word2Vec (2013) -- words as vectors!
- Limitation: No contextual understanding

**Era 3: Deep Learning NLP (2015-2017)**
- RNNs, LSTMs, GRUs
- Sequence-to-Sequence models
- Limitation: Sequential processing (slow), vanishing gradients

**Era 4: Transformers (2017-Present)**
- "Attention Is All You Need" paper (Vaswani et al., 2017)
- Parallel processing of tokens
- Self-attention mechanism
- Led to: BERT, GPT, T5, LLaMA, and more

**Era 5: Large Language Models (2020-Present)**
- Scaling laws: More data + More parameters = Better performance
- GPT-3 (175B params), GPT-4, LLaMA, Mistral, Gemma
- Emergent abilities: reasoning, code generation, translation

### 1.2 Transformer Architecture Overview

```
Input Text
    |
    v
[Tokenization] --> Convert text to token IDs
    |
    v
[Embedding Layer] --> Token IDs to dense vectors
    |
    v
[Positional Encoding] --> Add position information
    |
    v
[Multi-Head Self-Attention] --> Each token attends to all other tokens
    |                           Q (Query), K (Key), V (Value) matrices
    |                           Attention(Q,K,V) = softmax(QK^T / sqrt(d_k)) * V
    v
[Feed-Forward Network] --> Non-linear transformation
    |
    v
[Layer Norm + Residual] --> Stabilize training
    |
    v
  (Repeat N times)
    |
    v
[Output Layer] --> Probability distribution over vocabulary
    |
    v
Generated Token
```

**Key Insight -- Self-Attention:**
- Every token looks at every other token to decide what is important
- Example: "The cat sat on the mat because *it* was tired"
  - "it" attends strongly to "cat" (not "mat") -- this is attention!

### 1.3 Generative AI vs Traditional ML

| Aspect | Traditional ML | Generative AI (LLMs) |
|--------|---------------|---------------------|
| **Task** | Classification, Regression, Clustering | Text generation, Summarization, Translation, Code |
| **Input/Output** | Structured data --> Label/Number | Text --> Text (or Code, JSON, etc.) |
| **Training** | Supervised on labeled datasets | Self-supervised on massive text corpora |
| **Feature Engineering** | Manual feature extraction needed | Learns features automatically |
| **Flexibility** | One model per task | One model, many tasks (via prompting) |
| **Data Requirement** | Hundreds to thousands of examples | Billions of tokens for pre-training |
| **Example** | Spam classifier (SVM) | ChatGPT answering questions |

---
## Part 2: Setting Up Ollama & First LLM Call (Hands-On - 15 min)

### 2.1 Verify Ollama is Running
Make sure Ollama is running in the background. Open a terminal and run:
```bash
ollama serve
```
Then pull the model (if not already done):
```bash
ollama pull llama3.2
```

In [ ]:
# First, let's verify Ollama is accessible
import requests

try:
    response = requests.get("http://localhost:11434/api/tags")
    models = response.json()
    print("Ollama is running! Available models:")
    for model in models.get('models', []):
        print(f"  - {model['name']}")
except Exception as e:
    print(f"Error: Ollama is not running. Start it with 'ollama serve'\n{e}")

In [ ]:
# 2.2 Your First LLM Call using Ollama's REST API
import requests
import json

def call_ollama(prompt, model="llama3.2", temperature=0.7, max_tokens=500):
    """Make a direct API call to Ollama."""
    url = "http://localhost:11434/api/generate"
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": temperature,
            "num_predict": max_tokens
        }
    }
    response = requests.post(url, json=payload)
    return response.json()["response"]

# Test it!
result = call_ollama("What is a Large Language Model? Explain in 3 sentences.")
print(result)

In [ ]:
# 2.3 Using LangChain with Ollama (preferred method for the workshop)
from langchain_ollama import OllamaLLM

# Initialize the LLM
llm = OllamaLLM(
    model="llama3.2",
    temperature=0.7
)

# Simple invocation
response = llm.invoke("What is the difference between AI and Machine Learning?")
print(response)

---
## Part 3: Prompt Engineering Fundamentals (Theory + Hands-On - 30 min)

### 3.1 What is Prompt Engineering?

Prompt engineering is the art and science of designing input text (prompts) that guide an LLM to produce desired outputs. It is the primary way to "program" an LLM without changing its weights.

### Key Prompting Techniques:
1. **Zero-Shot Prompting** -- Ask directly, no examples
2. **Few-Shot Prompting** -- Provide examples before the question
3. **System Prompts** -- Set the role/behavior of the model
4. **Chain-of-Thought (CoT)** -- Ask the model to reason step-by-step
5. **Output Formatting** -- Request specific formats (JSON, tables, lists)

In [ ]:
# 3.2 Zero-Shot Prompting
# No examples, just a direct question

zero_shot_prompt = "Classify the sentiment of this text as Positive, Negative, or Neutral: 'The movie was absolutely fantastic, I loved every moment!'"

result = llm.invoke(zero_shot_prompt)
print("=== Zero-Shot Sentiment Classification ===")
print(result)

In [ ]:
# 3.3 Few-Shot Prompting
# Provide examples so the model learns the pattern

few_shot_prompt = """Classify the sentiment of each text as Positive, Negative, or Neutral.

Text: "The food was delicious and the service was great!"
Sentiment: Positive

Text: "The product broke after one day. Terrible quality."
Sentiment: Negative

Text: "The package arrived on time."
Sentiment: Neutral

Text: "I can't believe how bad the customer support was. Never buying again."
Sentiment:"""

result = llm.invoke(few_shot_prompt)
print("=== Few-Shot Sentiment Classification ===")
print(result)

In [ ]:
# 3.4 System Prompts (using ChatOllama for chat-style interaction)
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

chat_llm = ChatOllama(model="llama3.2", temperature=0.7)

# System prompt defines the LLM's role and behavior
messages = [
    SystemMessage(content="""You are a senior Python developer and code reviewer.
    When reviewing code:
    1. Identify bugs and issues
    2. Suggest improvements
    3. Rate the code quality from 1-10
    Be concise and technical."""),
    HumanMessage(content="""Review this Python code:

def fibonacci(n):
    if n <= 0:
        return 0
    elif n == 1:
        return 1
    else:
        return fibonacci(n-1) + fibonacci(n-2)

print(fibonacci(50))
""")
]

result = chat_llm.invoke(messages)
print("=== System Prompt: Code Reviewer ===")
print(result.content)

In [ ]:
# 3.5 Chain-of-Thought (CoT) Prompting
# Force the model to reason step-by-step

# Without CoT
simple_prompt = "If a shirt costs $25 and is on 20% discount, and tax is 8%, what is the final price?"
result_simple = llm.invoke(simple_prompt)
print("=== Without Chain-of-Thought ===")
print(result_simple)
print()

# With CoT
cot_prompt = """If a shirt costs $25 and is on 20% discount, and tax is 8%, what is the final price?

Think step by step:
1. Calculate the discount amount
2. Calculate the discounted price
3. Calculate the tax on the discounted price
4. Calculate the final price

Show your work for each step."""

result_cot = llm.invoke(cot_prompt)
print("=== With Chain-of-Thought ===")
print(result_cot)

---
## Part 4: Structured Output & Output Control (Hands-On - 25 min)

### 4.1 Generating Structured JSON Output
One of the most important skills -- making LLMs return data in a structured format.

In [ ]:
# 4.1 Structured JSON Output
import json

structured_prompt = """Extract the following information from the text and return ONLY valid JSON.

Text: "John Smith is a 28-year-old software engineer from Bangalore. He has 5 years of experience
in Python and Machine Learning. He previously worked at TCS and Infosys. His email is john@example.com."

Return JSON with these fields:
{
    "name": "",
    "age": 0,
    "role": "",
    "location": "",
    "experience_years": 0,
    "skills": [],
    "previous_companies": [],
    "email": ""
}

Return ONLY the JSON, no additional text."""

result = llm.invoke(structured_prompt)
print("=== Structured JSON Output ===")
print(result)

# Try to parse it
try:
    parsed = json.loads(result)
    print("\nParsed successfully!")
    print(f"Name: {parsed['name']}")
    print(f"Skills: {', '.join(parsed['skills'])}")
except json.JSONDecodeError as e:
    print(f"\nJSON parsing failed: {e}")
    print("Tip: You may need to clean the output or adjust the prompt")

In [ ]:
# 4.2 Temperature Control
# Temperature controls randomness: 0 = deterministic, 1+ = creative

prompt = "Write a one-line tagline for a coffee shop."

print("=== Temperature Comparison ===")
print()

# Low temperature (deterministic, focused)
llm_low = OllamaLLM(model="llama3.2", temperature=0.0)
print("Temperature = 0.0 (Deterministic):")
for i in range(3):
    result = llm_low.invoke(prompt)
    print(f"  Run {i+1}: {result.strip().split(chr(10))[0]}")

print()

# High temperature (creative, varied)
llm_high = OllamaLLM(model="llama3.2", temperature=1.5)
print("Temperature = 1.5 (Creative):")
for i in range(3):
    result = llm_high.invoke(prompt)
    print(f"  Run {i+1}: {result.strip().split(chr(10))[0]}")

In [ ]:
# 4.3 Max Tokens Control
# Control the length of the response

prompt = "Explain quantum computing."

print("=== Max Tokens = 50 ===")
result_short = call_ollama(prompt, max_tokens=50)
print(result_short)
print(f"\n[Length: ~{len(result_short.split())} words]")

print("\n=== Max Tokens = 200 ===")
result_long = call_ollama(prompt, max_tokens=200)
print(result_long)
print(f"\n[Length: ~{len(result_long.split())} words]")

In [ ]:
# 4.4 Output Format Control -- Making the LLM return specific formats

# Request a table format
table_prompt = """Compare Python, Java, and JavaScript in a markdown table with these columns:
- Language
- Typing
- Primary Use
- Learning Curve
- Performance

Return ONLY the markdown table."""

result = llm.invoke(table_prompt)
print("=== Table Format Output ===")
print(result)

In [ ]:
# 4.5 Using LangChain Prompt Templates for reusable prompts
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate

# Create a reusable prompt template
analysis_template = PromptTemplate(
    input_variables=["topic", "audience", "format"],
    template="""Explain {topic} for an audience of {audience}.

Requirements:
- Use simple language appropriate for the audience
- Include a real-world analogy
- Format the response as: {format}
- Keep it under 150 words"""
)

# Use the template with different inputs
prompt1 = analysis_template.format(
    topic="neural networks",
    audience="high school students",
    format="bullet points"
)
print("=== Neural Networks for High School Students ===")
print(llm.invoke(prompt1))

print("\n" + "="*50 + "\n")

prompt2 = analysis_template.format(
    topic="blockchain",
    audience="senior software engineers",
    format="a technical paragraph with key terminologies"
)
print("=== Blockchain for Engineers ===")
print(llm.invoke(prompt2))

In [ ]:
# 4.6 LangChain Chains -- Composing LLM calls
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# Create a chain: Prompt --> LLM --> Output Parser
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI tutor. Always explain concepts clearly with examples."),
    ("human", "Explain {concept} in the context of {domain}. Give a practical example.")
])

chain = prompt | chat_llm | StrOutputParser()

# Run the chain
result = chain.invoke({
    "concept": "recursion",
    "domain": "file system traversal"
})
print(result)

---
## Part 5: Exercises (10 min)

### Exercise 1: Build a Product Description Generator
Create a prompt that takes a product name and features, and generates a marketing description.

### Exercise 2: Multi-format Output
Given a paragraph about a company, extract information in THREE different formats:
- JSON
- Markdown table
- Bullet points

### Exercise 3: Prompt Comparison
Write the same question as zero-shot, few-shot, and CoT prompt. Compare the quality of outputs.

In [ ]:
# EXERCISE 1: Product Description Generator
# ==========================================
# TODO: Create a prompt template that generates marketing descriptions
# Input: product_name, features (list), target_audience
# Output: A compelling 50-word product description

product_prompt = PromptTemplate(
    input_variables=["product_name", "features", "target_audience"],
    template="""Write a compelling 50-word marketing description for the following product.

Product: {product_name}
Key Features: {features}
Target Audience: {target_audience}

The description should be engaging, highlight the main benefits, and include a call to action.
Return ONLY the description, nothing else."""
)

# Test it
prompt = product_prompt.format(
    product_name="SmartFit Pro Watch",
    features="Heart rate monitor, GPS tracking, 7-day battery, Water resistant",
    target_audience="Fitness enthusiasts aged 25-40"
)
print(llm.invoke(prompt))

In [ ]:
# EXERCISE 2: Multi-format Output
# ================================
# TODO: Extract information from this text in JSON, Table, and Bullet formats

company_text = """TechNova Solutions, founded in 2018 by Dr. Priya Sharma in Bangalore,
is an AI startup with 150 employees. They specialize in computer vision and NLP.
Revenue last year was $12M. Major clients include Flipkart and Swiggy.
They recently raised $25M in Series B funding."""

# JSON format
json_prompt = f"""Extract all factual information from this text and return as valid JSON:

{company_text}

Return ONLY valid JSON."""

print("=== JSON Format ===")
print(llm.invoke(json_prompt))

# Try the same for table and bullet points on your own!

In [ ]:
# EXERCISE 3: Prompt Comparison
# ==============================
# TODO: Ask "What causes seasons on Earth?" using three different techniques
# Compare the quality of outputs

# Zero-shot
print("=== Zero-Shot ===")
print(llm.invoke("What causes seasons on Earth?"))
print()

# Few-shot (provide examples of good explanations first)
# TODO: Write your few-shot prompt

# Chain-of-Thought
# TODO: Write your CoT prompt

---
## Summary

### Key Takeaways:
1. **LLMs evolved** from rule-based NLP through statistical methods to transformer-based architectures
2. **Attention mechanism** is the core innovation -- allowing tokens to attend to all other tokens
3. **Prompt engineering** is the primary way to control LLM behavior without retraining
4. **Zero-shot** works for simple tasks; **Few-shot** improves accuracy with examples; **CoT** helps with reasoning
5. **Temperature** controls creativity vs determinism; **Max tokens** controls response length
6. **Structured output** (JSON, tables) makes LLM responses programmable
7. **LangChain** provides templates and chains for building reusable LLM workflows

### Next Session: Open-Source LLMs with Ollama + Embeddings
We will dive deeper into how open-source models work, explore tokenization and embeddings, and build our first LLM-powered task.